<a href="https://colab.research.google.com/github/fawkes12345/LNDb-Multimodal/blob/main/notebooks/01_ct_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D CT Visualization and Nodule Localization

## Goal(with the help of SimpleITK)

The goal of this notebook is to connect the metadata in `allNods.csv`
with the actual 3D CT images.

For a selected pulmonary nodule, we will:

1. load the corresponding LNDb CT scan,
2. inspect image spacing and origin,
3. convert world coordinates into voxel coordinates,
4. locate the annotated pulmonary nodule,
5. visualize the nodule in axial, coronal, and sagittal views,
6. crop a 3D nodule patch for future neural-network input.

This step establishes the image modality pipeline before constructing
the image encoder.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/LNDb-Multimodal"
)

DATA_ROOT = PROJECT_ROOT / "data"
METADATA_DIR = DATA_ROOT / "metadata"
CT_DIR = DATA_ROOT / "ct"

CT_DIR.mkdir(parents=True, exist_ok=True)

print(CT_DIR)

Mounted at /content/drive
/content/drive/MyDrive/LNDb-Multimodal/data/ct


In [2]:
!pip install SimpleITK -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.1 MB/s eta 0:00:00


In [3]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
all_nodules_df = pd.read_csv(
    METADATA_DIR / "allNods.csv"
)

example = all_nodules_df[
    (all_nodules_df["LNDbID"] == 1)
    &
    (all_nodules_df["FindingID"] == 1)
].iloc[0]

display(example)

,0
LNDbID,1
RadID,"1,2,3"
RadFinding,"1,1,1"
FindingID,1
Nodule,"1,1,1"
x,-44.203451
y,-119.073242
z,-37.5
DiamEq_Rad,7.516572
Texture,5.0


In [5]:
world_coord = (
    float(example["x"]),
    float(example["y"]),
    float(example["z"])
)

print("World coordinate:", world_coord)

World coordinate: (-44.20345052166667, -119.07324219166668, -37.5)
